In [2]:
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, QuantileTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score, precision_recall_curve,
                             precision_score, recall_score, f1_score)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src' / 'evaluation.py').exists())
path = ROOT / 'data'
results = ROOT / 'results'; results.mkdir(exist_ok=True)
SEED = 42

DROP = ['company', 'industry', 'link', 'num', 'emis_id', 'sector_2', 'sector_3', 'sector_4',
        'Revenue/employee', 'Fixed_assets/employee', 'EBITDA/cash_flow']
NOT_FEATURES = ['operational_status', 'year', 'main_label']

In [3]:
def load_horizon(h):
    """h = years before distress (0-5). File h1 = 0 years, h2 = 1 year, ..."""
    df = pd.read_parquet(path/f'company_years_h{h+1}.parquet')
    y = df.main_label.astype(int).to_numpy()
    groups = df.company.to_numpy()
    X = df.drop(columns=DROP + NOT_FEATURES)
    non_num = X.select_dtypes(exclude='number').columns.tolist()
    if non_num:
        print('dropping non-numeric:', non_num)
    X = X.select_dtypes('number').astype('float32')
    return X, y, groups

X, y, groups = load_horizon(1)
print(X.shape, y.sum())

dropping non-numeric: ['has_multiple_industries']
(996500, 128) 3054


In [4]:
def make_folds(y, groups, n=5, seed=SEED):
    fold = np.full(len(y), -1)
    sgkf = StratifiedGroupKFold(n_splits=n, shuffle=True, random_state=seed)
    for k, (_, test_idx) in enumerate(sgkf.split(np.zeros(len(y)), y, groups)):
        fold[test_idx] = k
    return fold

fold = make_folds(y, groups)

# checks
assert (fold >= 0).all()
assert pd.Series(fold).groupby(groups).nunique().max() == 1, 'a company is in two folds!'
display(pd.crosstab(fold, y, margins=True))

col_0,0,1,All
row_0,,,
0,198690,611,199301
1,198689,610,199299
2,198688,611,199299
3,198690,611,199301
4,198689,611,199300
All,993446,3054,996500


In [5]:
def best_threshold(y_true, score):
    """Threshold that maximises F1 - chosen on the validation fold only."""
    p, r, t = precision_recall_curve(y_true, score)
    f1 = 2 * p * r / (p + r + 1e-12)
    return t[np.argmax(f1[:-1])]

def evaluate(y_true, score, thr):
    pred = (score >= thr).astype(int)
    return dict(roc_auc=roc_auc_score(y_true, score),
                pr_auc=average_precision_score(y_true, score),
                pr_baseline=y_true.mean(),
                precision=precision_score(y_true, pred, zero_division=0),
                recall=recall_score(y_true, pred),
                f1=f1_score(y_true, pred),
                threshold=thr)

In [6]:
def best_threshold(y_true, score):
    """Threshold that maximises F1 - chosen on the validation fold only."""
    p, r, t = precision_recall_curve(y_true, score)
    f1 = 2 * p * r / (p + r + 1e-12)
    return t[np.argmax(f1[:-1])]

def evaluate(y_true, score, thr):
    pred = (score >= thr).astype(int)
    return dict(roc_auc=roc_auc_score(y_true, score),
                pr_auc=average_precision_score(y_true, score),
                pr_baseline=y_true.mean(),
                precision=precision_score(y_true, pred, zero_division=0),
                recall=recall_score(y_true, pred),
                f1=f1_score(y_true, pred),
                threshold=thr)

In [7]:
def run_cv(name, h, X, y, fold, make_model=None, score_fn=None):
    """Fold k = test, fold k+1 = validation (threshold), the other 3 = training."""
    rows = []
    for k in range(5):
        te, va = fold == k, fold == (k + 1) % 5
        tr = ~(te | va)
        if make_model is not None:
            m = make_model().fit(X[tr], y[tr])
            s_va, s_te = m.predict_proba(X[va])[:, 1], m.predict_proba(X[te])[:, 1]
        else:
            s_va, s_te = score_fn(X[tr], X[va], X[te])
        thr = best_threshold(y[va], s_va)
        rows.append({'model': name, 'horizon': h, 'fold': k, **evaluate(y[te], s_te, thr)})
        print(f'{name} fold {k}: ROC-AUC {rows[-1]["roc_auc"]:.3f}')
    return pd.DataFrame(rows)

In [8]:
res_dummy = run_cv('dummy', 1, X, y, fold, make_model=lambda: DummyClassifier(strategy='prior'))

dummy fold 0: ROC-AUC 0.500
dummy fold 1: ROC-AUC 0.500
dummy fold 2: ROC-AUC 0.500
dummy fold 3: ROC-AUC 0.500
dummy fold 4: ROC-AUC 0.500


In [9]:
ALTMAN = {'Working_capital/total_assets': 0.717, 'Retained_profit/total_assets': 0.847,
          'EBIT/total_assets': 3.107, 'Equity/total_liabilities': 0.420,
          'Total_operating_revenue/total_assets': 0.998}
print('missing:', [c for c in ALTMAN if c not in X.columns])   # must be []

def altman_scores(Xtr, Xva, Xte):
    cols, w = list(ALTMAN), np.array(list(ALTMAN.values()))
    med = Xtr[cols].median()                       # fill gaps with TRAIN medians only
    z = lambda X_: X_[cols].fillna(med).to_numpy() @ w
    return -z(Xva), -z(Xte)                        # low Z = high risk, so use -Z as the score

res_altman = run_cv('altman_z', 1, X, y, fold, score_fn=altman_scores)

missing: []
altman_z fold 0: ROC-AUC 0.773
altman_z fold 1: ROC-AUC 0.765
altman_z fold 2: ROC-AUC 0.768
altman_z fold 3: ROC-AUC 0.757
altman_z fold 4: ROC-AUC 0.750


In [11]:
def lr_standard():   # like the paper: median fill + standard scaling
    return make_pipeline(SimpleImputer(strategy='median'),
                         StandardScaler(),
                         LogisticRegression(class_weight='balanced', C=0.1, max_iter=2000))

def lr_robust():     # missing indicators + quantile transform to tame outliers
    return make_pipeline(SimpleImputer(strategy='median', add_indicator=True),
                         QuantileTransformer(output_distribution='normal', n_quantiles=1000,
                                             subsample=200_000, random_state=SEED),
                         LogisticRegression(class_weight='balanced', C=0.1, max_iter=2000))

res_lr1 = run_cv('logreg_standard', 1, X, y, fold, make_model=lr_standard)
res_lr2 = run_cv('logreg_robust',   1, X, y, fold, make_model=lr_robust)

KeyboardInterrupt: 

In [12]:
res = pd.concat([res_dummy, res_altman, res_lr1, res_lr2])
res.to_csv(results/'03_baselines_h1.csv', index=False)

cols = ['roc_auc', 'pr_auc', 'pr_baseline', 'precision', 'recall', 'f1']
display(res.groupby('model')[cols].agg(['mean', 'std']).round(3))

NameError: name 'res_lr1' is not defined

In [13]:
# 1. Is robust better in every fold? (same folds -> paired comparison)
p = res.pivot(index='fold', columns='model', values='roc_auc')
display((p.logreg_robust - p.logreg_standard).round(3))

NameError: name 'res' is not defined